# Data Preprocessing Template

## 1. Importing the libraries

In [ ]:
pip install scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

## 2. Importing the dataset

Load the CSV, then split into:
- `X` — the features/independent variables (all columns except the last: Country, Age, Salary)
- `y` — the target/dependent variable (last column: Purchased)

`.iloc[:, :-1]` grabs all rows and all columns except the last; `.iloc[:, -1]` grabs just the last column. We convert to `.values` so we get plain numpy arrays, matching what sklearn expects downstream.

In [ ]:
dataset = pd.read_csv('Preprocess_template_data.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

print(X)
print(y)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 nan]
 ['France' 35.0 58000.0]
 ['Spain' nan 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]
<StringArray>
['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes']
Length: 10, dtype: str


## 3. Taking care of missing data

Looking at the CSV: row for Germany/40 has a missing `Salary`, and the Spain row has a missing `Age`. Rather than dropping those rows (losing data), we impute — fill the gap with the column's mean, using sklearn's `SimpleImputer`.

- `missing_values=np.nan` tells it what a "missing" marker looks like (pandas reads empty CSV cells as `NaN`).
- `strategy='mean'` fills gaps with the column average.
- We only fit/transform columns 1 and 2 (`Age`, `Salary`) — column 0 (`Country`) is text, so a numeric mean doesn't apply.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputer.fit(X[:, 1:3])
X[:, 1:3] = imputer.transform(X[:, 1:3])

print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 63777.77777777778]
 ['France' 35.0 58000.0]
 ['Spain' 38.77777777777778 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


## 4. Encoding categorical data

Models need numbers, not text. `Country` and `Purchased` are both categorical, but they need *different* encodings depending on their role:

### Encoding the Independent Variable (`Country`)

`Country` has no natural order (France isn't "more" or "less" than Spain), so a plain 0/1/2 label would wrongly imply an ordering the model could exploit. Instead we **one-hot encode**: each category becomes its own binary column.

We use `ColumnTransformer` to apply `OneHotEncoder` to column 0 only, and pass the rest of the columns through unchanged (`remainder='passthrough'`). The result is wrapped back into a numpy array with `np.array(...)`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

print(X)

[[1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [0.0 1.0 0.0 30.0 54000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 35.0 58000.0]
 [0.0 0.0 1.0 38.77777777777778 52000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 37.0 67000.0]]


### Encoding the Dependent Variable (`Purchased`)

`y` is binary (`Yes`/`No`) and is the *target*, not a feature — the model treats it as a single output to predict, so there's no risk of a false ordering being "used" the way there was with `Country`. A simple `LabelEncoder` mapping `No → 0`, `Yes → 1` is enough.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(y)

[0 1 0 0 1 1 0 1 0 1]


## 5. Splitting the dataset into the Training set and Test set

We hold out a portion of the data (here 20%) that the model never sees during training, so we can later evaluate it on genuinely unseen examples. `random_state=1` just fixes the shuffle so the split is reproducible across runs.

Important ordering note: splitting happens **before** feature scaling, so scaling stats are computed only from the training set — not leaked from the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(X_train)
print(X_test)
print(y_train)
print(y_test)

[[0.0 0.0 1.0 38.77777777777778 52000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 35.0 58000.0]]
[[0.0 1.0 0.0 30.0 54000.0]
 [1.0 0.0 0.0 37.0 67000.0]]
[0 1 0 0 1 1 0 1]
[0 1]


## 6. Feature Scaling

`Age` (roughly 27–50) and `Salary` (roughly 48000–83000) live on very different scales. Distance-based models (e.g. k-NN, SVM) would let `Salary` dominate purely because its numbers are bigger — not because it's actually more important. Standardization fixes this by rescaling each feature to mean 0, std 1: `z = (x - mean) / std`.

We **fit** the scaler on the training set only (`fit_transform`) and **only transform** the test set (`transform`, no fit) — reusing the training mean/std. This avoids leaking any information about the test set into how we scale.

Note: the one-hot encoded country columns are already 0s and 1s, so we don't scale those — only the numeric `Age`/`Salary` columns (index 3 onward after encoding).

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train[:, 3:] = sc.fit_transform(X_train[:, 3:])
X_test[:, 3:] = sc.transform(X_test[:, 3:])

print(X_train)
print(X_test)

[[0.0 0.0 1.0 -0.19159184384578545 -1.0781259408412425]
 [0.0 1.0 0.0 -0.014117293757057777 -0.07013167641635372]
 [1.0 0.0 0.0 0.566708506533324 0.633562432710455]
 [0.0 0.0 1.0 -0.30453019390224867 -0.30786617274297867]
 [0.0 0.0 1.0 -1.9018011447007988 -1.420463615551582]
 [1.0 0.0 0.0 1.1475343068237058 1.232653363453549]
 [0.0 1.0 0.0 1.4379472069688968 1.5749910381638885]
 [1.0 0.0 0.0 -0.7401495441200351 -0.5646194287757332]]
[[0.0 1.0 0.0 -1.4661817944830124 -0.9069571034860727]
 [1.0 0.0 0.0 -0.44973664397484414 0.2056403393225306]]


## 7. Train Models

`X_train`, `X_test`, `y_train`, `y_test` are now fully preprocessed (imputed, encoded, split, scaled) and ready to feed into a classifier. This is a placeholder — tell me which model(s) you want to try (e.g. Logistic Regression, k-NN, SVM) and we'll add the training/evaluation cells here.